In [1]:
import math
from statistics import NormalDist

import numpy as np

def prom_update(prom_prev: float, x_new: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return x_new
    else:
        return prom_prev + (x_new - prom_prev) / n_curr


def s_cuad_update(s_cuad_prev: float, prom_prev: float, prom_curr: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return 0.0
    else:
        return (1 - (1/(n_curr - 1))) * s_cuad_prev + n_curr * ((prom_curr - prom_prev)**2)

def calculo_z_standard(alpha: float):
    return NormalDist().inv_cdf(1 - alpha / 2)


def intervalo(prom: float, s_cuad: float, n: int, alpha: float):
    z_alpha_2 = calculo_z_standard(alpha)

    std = math.sqrt(s_cuad/n)
    izq = prom - z_alpha_2 * std
    der = prom + z_alpha_2 * std

    return izq, der

### Ejercicio 10.
Considerar un sistema con dos servidores en paralelo, cada uno con su propia cola de espera. Al arribar, un cliente se incorpora a la cola más corta. En caso de que ambos servidores tengan igual cantidad de clientes en espera, incluyendo el caso en que ambos servidores estén libres, el cliente se incorpora a la cola del servidor 1.

Suponer que los tiempos de servicio del servidor 1 y del servidor 2 se distribuyen exponencialmente con tasas $3$ y $4$ clientes por hora, respectivamente. Los clientes llegan de acuerdo a un proceso de Poisson no homogéneo con función de intensidad $\lambda$ dada por:
$$ \lambda(t) = 7 - \frac{1}{t + 1}, \quad \text{donde t se mide en horas.} $$

a. Desarrollar un programa que simule el proceso, registrando los tiempos de llegada, los tiempos de servicio, las salidas desde cada servidor y la evolución del número de trabajos en cola en cada servidor.

b. Utilizar el programa para estimar el tiempo promedio de permanencia en el sistema de los primeros $1000$ clientes. Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.01$.

c. Estimar la número esperado de servicios realizados por el servidor 1 entre los primeros $1000$ servicios completados por el sistema. Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.1$.

d. Construir intervalos de confianza del $90 \%$ para las cantidades estimadas en los incisos anteriores.

---

In [2]:
# Punto A

def lamda_t(t: float):
    if t < 0:
        raise ValueError("t debe ser mayor o igual a 0, ya que las horas son números positivos")
    
    return 7 - 1 / (t + 1)

def t_llega_cliente():
    lamda_cota = 7  # lamda_t tiene asíntota en 7
    t = 0

    while True:
        t -= math.log(1 - np.random.random()) / lamda_cota
        V = np.random.random()

        if V < lamda_t(t) / lamda_cota:
            return t

def t_servicio_s1():
    return -math.log(1 - np.random.random()) / 3

def t_servicio_s2():
    return -math.log(1 - np.random.random()) / 4

def punto_a(n_servicios_completados: int = 1_000):
    # Registros
    A = []
    D = []
    N = [0]
    N_T = [0]

    # Inicialización
    cola = []
    t = 0
    NA, C1, C2 = 0, 0, 0
    n, i1, i2 = 0, 0, 0
    
    tA = t_llega_cliente()
    t1 = math.inf
    t2 = math.inf
    
    while tA < math.inf or t1 < math.inf or t2 < math.inf:
        evento_prox = min(tA, t1, t2)
        
        if len(D) >= n_servicios_completados:
            d = [el[0] for el in D]
            d.sort()
            
            d = d[:n_servicios_completados]
            
            primeros_n_serv_comp = all(num == i for i, num in enumerate(d, start=1))
            if primeros_n_serv_comp:
                break
        
        if evento_prox == tA:
            t = tA
            NA += 1
            tA = t + t_llega_cliente()
        
            if i1 == 0:
                i1 = NA
                t1 = t + t_servicio_s1()
            elif i2 == 0:
                i2 = NA
                t2 = t + t_servicio_s2()
            else:
                cola.append(NA)
            
            n += 1
            A.append(t)
            N.append(n)
            N_T.append(t)
        
        elif evento_prox == t1:
            t = t1
            C1 += 1
            n -= 1

            n_cliente = i1
            if len(cola) > 0:
                i1 = cola.pop(0)
                t1 = t + t_servicio_s1()
            else:
                i1 = 0
                t1 = math.inf

            D.append((n_cliente, 1, t))
            N.append(n)
            N_T.append(t)

        else:
            t = t2
            C2 += 1
            n -= 1

            n_cliente = i2
            if len(cola) > 0:
                i2 = cola.pop(0)
                t2 = t + t_servicio_s2()
            else:
                i2 = 0
                t2 = math.inf

            D.append((n_cliente, 2, t))
            N.append(n)
            N_T.append(t)
    
    return t, A, D, N, N_T

In [3]:
# Punto B

def generar_X_i_b():
    servicios_completados = 1_000

    _, A, D, _, _ = punto_a(servicios_completados)
    
    D.sort(key=lambda x: x[0])
    D = D[:servicios_completados]

    s = 0
    for i in range(servicios_completados):
        n_cliente = D[i][0] - 1
        
        s += D[i][2] - A[n_cliente]

    return s / servicios_completados

def punto_b():
    media = generar_X_i_b()
    s_cuad, n = 0, 1

    while n < 100 or math.sqrt(s_cuad / n) >= 0.01:
        X = generar_X_i_b()
        n += 1

        media_prev = media
        media = prom_update(media, X, n)
        s_cuad = s_cuad_update(s_cuad, media_prev, media, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Tiempo promedio de permanencia en el sistema de los primeros 1000 clientes: {media:.4f} horas")
    print(f"Desviación estándar muestral = {math.sqrt(s_cuad / n):.4f} ")

punto_b()

Cantidad de datos generados:  2109
Tiempo promedio de permanencia en el sistema de los primeros 1000 clientes: 1.1401 horas
Desviación estándar muestral = 0.0100 


In [4]:
# Punto C

def generar_X_i_c():
    servicios_completados = 1_000

    _, _, D, _, _ = punto_a(servicios_completados)

    D.sort(key=lambda x: x[0])
    D = D[:servicios_completados]

    count = 0
    for i in range(servicios_completados):
        if D[i][1] == 1:
            count +=1

    return count / servicios_completados

def punto_c():
    p = generar_X_i_c()
    n = 1

    while n < 100 or math.sqrt(p * (1 - p) / n) >= 0.1:
        X = generar_X_i_c()
        n += 1

        p = prom_update(p, X, n)

    print(f"Cantidad de datos generados: ", n)
    print(f"Probabilidad de que el servicio sea realizado por el servidor 1: {p:.4f}")
    print(f"Desviación estándar muestral = {math.sqrt(p * (1 - p) / n):.4f} ")


punto_c()

Cantidad de datos generados:  100
Probabilidad de que el servicio sea realizado por el servidor 1: 0.4423
Desviación estándar muestral = 0.0497 


In [6]:
# Punto D

def sim_media(l: float, alpha: float):
    # l = semi-ancho = L / 2

    z_alpha_2 = calculo_z_standard(alpha)
    cota = l / (z_alpha_2 * 2)

    promedio = generar_X_i_b()
    s_cuad, n = 0, 1

    while n <= 100 or math.sqrt(s_cuad / n) >= cota:
        X = generar_X_i_b()
        n += 1

        promedio_prev = promedio
        promedio = prom_update(promedio, X, n)
        s_cuad = s_cuad_update(s_cuad, promedio_prev, promedio, n)

    i_d, i_i = intervalo(promedio, s_cuad, n, alpha)

    print(
        f"{"B":>6} | {l:>9} | {n:9d} | {f"[{i_d:.4f}, {i_i:.4f}]":>18} | {math.sqrt(s_cuad / n):10.4f} | {promedio:10.4f}")


def sim_prob_c(l: float, alpha: float):
    z_alpha_2 = calculo_z_standard(alpha)
    cota = l / (z_alpha_2 * 2)

    p = generar_X_i_c()
    n = 1

    while n <= 100 or math.sqrt(p*(1 - p) / n) >= cota:
        X = generar_X_i_c()
        n += 1

        p = prom_update(p, X, n)

    i_d, i_i = intervalo(p, math.sqrt(p*(1 - p) / n), n, alpha)

    print(
        f"{"C":>6} | {l:>9} | {n:9d} | {f"[{i_d:.4f}, {i_i:.4f}]":>18} | {math.sqrt(p*(1 - p) / n):10.4f} | {p:10.4f}")


l = 0.1
alpha = 0.1
print(f"{"Punto":>6} | {"Amplitud":>9} | {"N° Sim":>9} | {"Intervalo obtenido":>18} | {"STDM":>10} | {"Aprox (media/p)":>10}")
sim_media(l, alpha)
sim_prob_c(l, alpha)

 Punto |  Amplitud |    N° Sim | Intervalo obtenido |       STDM | Aprox (media/p)
     B |       0.1 |       296 |   [1.1141, 1.2139] |     0.0303 |     1.1640
     C |       0.1 |       268 |   [0.4274, 0.4624] |     0.0304 |     0.4449
